In [1]:
import pandas as pd 

chunks_path = "all_corpus_final_chunks.parquet"
chunk_df = pd.read_parquet(chunks_path)

chunk_df.shape, chunk_df.head()

((301184, 6),
    chunk_id ticker chunk_type                               section_title  \
 0         0      A       text                                    Overview   
 1         1      A       text                                    Overview   
 2         2      A       text  Life Sciences and Applied Markets Business   
 3         3      A       text           Life Sciences and Applied Markets   
 4         4      A       text           Life Sciences and Applied Markets   
 
                                                 text  \
 0  Agilent Technologies Inc. ("we", "Agilent" or ...   
 1  and life sciences research areas to interrogat...   
 2  Our life sciences and applied markets business...   
 3  The Pharmaceutical, Biopharmaceutical, CRO & C...   
 4  Our products are used to test for safety, qual...   
 
                                       embedding_text  
 0  Overview\n\nAgilent Technologies Inc. ("we", "...  
 1  Overview\n\nand life sciences research areas t...  
 2  

In [2]:
from sentence_transformers import SentenceTransformer

embedding_model_name = "BAAI/bge-small-en-v1.5"

embedding_model = SentenceTransformer(embedding_model_name)

embedding_model

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2940.73it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)

In [3]:
sample_texts = chunk_df["embedding_text"].head(5).tolist()

sample_embeddings = embedding_model.encode(
    sample_texts,
    batch_size=5,
    show_progress_bar=True,
    normalize_embeddings=True,
)

sample_embeddings.shape

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.40it/s]


(5, 384)

In [5]:
num_chunks = len(chunk_df)
embedding_dim = sample_embeddings.shape[1]

estimated_size_gb = num_chunks*embedding_dim*4/(1024**3)

num_chunks, embedding_dim, estimated_size_gb

(301184, 384, 0.43084716796875)

In [7]:
embedding_texts = chunk_df["embedding_text"].tolist()

embeddings = embedding_model.encode(
    embedding_texts,
    batch_size=512,
    show_progress_bar=True,
    normalize_embeddings=True,
)

embeddings.shape

Batches: 100%|██████████| 589/589 [11:03<00:00,  1.13s/it]


(301184, 384)

In [8]:
import numpy as np

embeddings = embeddings.astype("float32")

np.save("all_corpus_embeddings.npy", embeddings)

embeddings.shape, embeddings.dtype

((301184, 384), dtype('float32'))

In [2]:
import numpy as np
import pandas as pd

chunk_df = pd.read_parquet("all_corpus_final_chunks.parquet")
embeddings = np.load("all_corpus_embeddings.npy")

chunk_df.shape, embeddings.shape, embeddings.dtype

((301184, 6), (301184, 384), dtype('float32'))

In [5]:
assert len(chunk_df) == embeddings.shape[0]
assert chunk_df["chunk_id"].is_unique
assert chunk_df["chunk_id"].iloc[0] == 0
assert chunk_df["chunk_id"].iloc[-1] == len(chunk_df) -1
assert embeddings.dtype == "float32"

len(chunk_df), embeddings.shape

(301184, (301184, 384))

In [7]:
import faiss

embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

index.add(embeddings)
index.ntotal

301184

In [8]:
faiss.write_index(index, "all_corpus_faiss.index")